In [16]:
import sys
from pathlib import Path

project_root = Path().resolve().parent
sys.path.append(str(project_root))

In [17]:
import json
from models.llm_ner import get_entities_from_llm
from models.llm_text_embeddings import get_embeddings
from models.metric import CDE, exhaustive_CDE, EF, CDEF

In [18]:
def parse_response(response:str):
    response = json.loads(response)
    
    entities = []

    for e in response['entities']:
        entity = e['entity']
        type = e['type']
        entities.append([entity, type])

    return entities

In [19]:
def run_test_on_use_case(types, sentence, gold_entities, verbose=True):
    
    response = get_entities_from_llm(sentence, types)
    generated_entities = parse_response(response)

    if verbose:
        print(f'{"SENTENCE:":20s} {sentence}')
        print(f'{"GOLD_ENTITIES:":20s} {gold_entities}')
        print(f'{"GENERATED_ENTITIES:":20s} {generated_entities}')

    gold_embeddings = get_embeddings(gold_entities)
    generated_embeddings = get_embeddings(generated_entities)
    
    cde=CDE(gold_embeddings, generated_embeddings)
    exh_cde=exhaustive_CDE(gold_embeddings, generated_embeddings)
    ef=EF(gold_embeddings, generated_embeddings)
    cdef_05=CDEF(gold_embeddings, generated_embeddings, beta=0.5)
    cdef_1=CDEF(gold_embeddings, generated_embeddings, beta=1)
    cdef_15=CDEF(gold_embeddings, generated_embeddings, beta=1.5)

    if verbose:
        print(f'{"CDE:":10s}{cde}')
        print(f'{"Exh_CDE":10s}{exh_cde}')
        print(f'{"EF:":10s}{ef}')
        print(f'{"CDEF-0.5":10s}{cdef_05}')
        print(f'{"CDEF-1.0:":10s}{cdef_1}')
        print(f'{"CDEF-1.5:":10s}{cdef_15}')

    return cde, exh_cde, ef, cdef_05, cdef_1, cdef_15

# Use Cases
Three types of entities:
- Symptom,
- Drug,
- Disease.

In [20]:
types = ['Symptom', 'Drug', 'Disease']

## 1. Discontinuous Entities

### UC1.1 
**The patient reported pain in the lower back and occasionally in the right leg.**
- pain in the lower back; Symptom
- pain in the right leg; Symptom

In [21]:
sentence = "The patient reported pain in the lower back and occasionally in the right leg."
gold_entities = [
    ['pain in the lower back', 'Symptom'],
    ['pain in the right leg', 'Symptom']
]

cde, exh_cde, ef, cdef_05, cdef_1, cdef_15 = run_test_on_use_case(types, sentence, gold_entities)

SENTENCE:            The patient reported pain in the lower back and occasionally in the right leg.
GOLD_ENTITIES:       [['pain in the lower back', 'Symptom'], ['pain in the right leg', 'Symptom']]
GENERATED_ENTITIES:  [['pain', 'Symptom'], ['lower back', 'Symptom'], ['right leg', 'Symptom']]
CDE:      0.1215967271289205
Exh_CDE   0.1215967271289205
EF:       0.19999999999999996
CDEF-0.5  0.9076162206432932
CDEF-1.0: 0.8640301313059162
CDEF-1.5: 0.8382264068376933


### UC1.2
**The patient experienced shortness of breath on exertion and sometimes at rest.**
- shortness of breath on exertion; Symptom
- shortness of breath at rest; Symptom

In [22]:
sentence = "The patient experienced shortness of breath on exertion and sometimes at rest."
gold_entities = [
    ['shortness of breath on exertion', 'Symptom'],
    ['shortness of breath at rest', 'Symptom']
]

cde, exh_cde, ef, cdef_05, cdef_1, cdef_15 = run_test_on_use_case(types, sentence, gold_entities)

SENTENCE:            The patient experienced shortness of breath on exertion and sometimes at rest.
GOLD_ENTITIES:       [['shortness of breath on exertion', 'Symptom'], ['shortness of breath at rest', 'Symptom']]
GENERATED_ENTITIES:  [['shortness of breath', 'Symptom']]
CDE:      0.11803836561393666
Exh_CDE   0.11803836561393666
EF:       -0.33333333333333337
CDEF-0.5  0.8694315999580503
CDEF-1.0: 0.7804205226499786
CDEF-1.5: 0.7323579058696681


### UC1.3
**Have stopped taking and immediatedly muscle/joint pain eased and sleeping much better.**
- muscle pain; Symptom
- joint pain; Symptom

In [23]:
sentence = "Have stopped taking and immediatedly muscle/joint pain eased and sleeping much better."
gold_entities = [
    ['muscle pain', 'Symptom'],
    ['joint pain', 'Symptom']
]

cde, exh_cde, ef, cdef_05, cdef_1, cdef_15 = run_test_on_use_case(types, sentence, gold_entities)

SENTENCE:            Have stopped taking and immediatedly muscle/joint pain eased and sleeping much better.
GOLD_ENTITIES:       [['muscle pain', 'Symptom'], ['joint pain', 'Symptom']]
GENERATED_ENTITIES:  [['muscle/joint pain', 'Symptom']]
CDE:      0.10130145729433637
Exh_CDE   0.10130145729433637
EF:       -0.33333333333333337
CDEF-0.5  0.8751337453753893
CDEF-1.0: 0.7832837527714838
CDEF-1.5: 0.7339071490086659


### UC1.4
**Menstrual cramps present with or without vaginal bleeding.**
- menstrual cramps with vaginal bleeding; Symptom
- menstrual cramps without vaginal bleeding; Symptom

In [24]:
sentence = "Menstrual cramps present with or without vaginal bleeding."
gold_entities = [
    ['menstrual cramps with vaginal bleeding', 'Symptom'],
    ['menstrual cramps without vaginal bleeding', 'Symptom']
]
cde, exh_cde, ef, cdef_05, cdef_1, cdef_15 = run_test_on_use_case(types, sentence, gold_entities)

SENTENCE:            Menstrual cramps present with or without vaginal bleeding.
GOLD_ENTITIES:       [['menstrual cramps with vaginal bleeding', 'Symptom'], ['menstrual cramps without vaginal bleeding', 'Symptom']]
GENERATED_ENTITIES:  [['menstrual cramps', 'Symptom'], ['vaginal bleeding', 'Symptom']]
CDE:      0.14882194156765166
Exh_CDE   0.14882194156765166
EF:       0.0
CDEF-0.5  0.9395719209682687
CDEF-1.0: 0.9613567746519021
CDEF-1.5: 0.9758607777062375


### UC1.5
**After the second pill the same progression of symptoms only now the abdominal gas,cramps and pain would be with me all day.**
- abdominal gas; Symptom
- abdominal cramps; Symptom
- abdominal pain; Symptom

In [25]:
sentence = "After the second pill the same progression of symptoms only now the abdominal gas,cramps and pain would be with me all day."
gold_entities = [
    ['abdominal gas', 'Symptom'],
    ['abdominal cramps', 'Symptom'],
    ['abdominal pain', 'Symptom']
]

cde, exh_cde, ef, cdef_05, cdef_1, cdef_15 = run_test_on_use_case(types, sentence, gold_entities)

SENTENCE:            After the second pill the same progression of symptoms only now the abdominal gas,cramps and pain would be with me all day.
GOLD_ENTITIES:       [['abdominal gas', 'Symptom'], ['abdominal cramps', 'Symptom'], ['abdominal pain', 'Symptom']]
GENERATED_ENTITIES:  [['abdominal gas', 'Symptom'], ['cramps', 'Symptom'], ['pain', 'Symptom']]
CDE:      0.1263790903944034
Exh_CDE   0.1263790903944034
EF:       0.0
CDEF-0.5  0.948801319576102
CDEF-1.0: 0.9673744299342727
CDEF-1.5: 0.9796675889981112


## 2. Long, Descriptive Entities

### UC2.1
**Without this drug I was severly restricted and could only walk less than 100 meters.**
- could only walk less than 100 meters; Symptom

In [26]:
sentence = "Without this drug I was severly restricted and could only walk less than 100 meters."
gold_entities = [
    ['could only walk less than 100 meters', 'Symptom']
]

cde, exh_cde, ef, cdef_05, cdef_1, cdef_15 = run_test_on_use_case(types, sentence, gold_entities)

SENTENCE:            Without this drug I was severly restricted and could only walk less than 100 meters.
GOLD_ENTITIES:       [['could only walk less than 100 meters', 'Symptom']]
GENERATED_ENTITIES:  [['this drug', 'Drug']]
CDE:      2.0
Exh_CDE   2.0
EF:       0.0
CDEF-0.5  0.0
CDEF-1.0: 0.0
CDEF-1.5: 0.0


### UC2.2
**Recently I've been feeling like my stomach is full and empty at the same time hard to explain.**
- feeling stomach is full and empty at the same time; Symptom

In [27]:
sentence = "Recently I've been feeling like my stomach is full and empty at the same time hard to explain."
gold_entities = [
    ['feeling stomach is full and empty at the same time', 'Symptom']
]

cde, exh_cde, ef, cdef_05, cdef_1, cdef_15 = run_test_on_use_case(types, sentence, gold_entities)

SENTENCE:            Recently I've been feeling like my stomach is full and empty at the same time hard to explain.
GOLD_ENTITIES:       [['feeling stomach is full and empty at the same time', 'Symptom']]
GENERATED_ENTITIES:  [['stomach', 'Disease'], ['full', 'Symptom'], ['empty', 'Symptom']]
CDE:      0.3367578766208573
Exh_CDE   0.3367578766208573
EF:       0.5
CDEF-0.5  0.7342270059904835
CDEF-1.0: 0.6245178043627546
CDEF-1.5: 0.5699283948471008


### UC2.3
**The boy was diagnosed with a very bad lung infection that affected the lower part of his right lung.**
- lung infection of the lower part of right lung; Disease

In [28]:
sentence = "The boy was diagnosed with a very bad lung infection that affected the lower part of his right lung."
gold_entities = [
    ['lung infection of the lower part of right lung', 'Disease']
]

cde, exh_cde, ef, cdef_05, cdef_1, cdef_15 = run_test_on_use_case(types, sentence, gold_entities)

SENTENCE:            The boy was diagnosed with a very bad lung infection that affected the lower part of his right lung.
GOLD_ENTITIES:       [['lung infection of the lower part of right lung', 'Disease']]
GENERATED_ENTITIES:  [['lung infection', 'Disease']]
CDE:      0.18487491599217287
Exh_CDE   0.18487491599217287
EF:       0.0
CDEF-0.5  0.924657132982076
CDEF-1.0: 0.9515415846345043
CDEF-1.5: 0.9696130899642383


### UC2.4
**The doctors started strong medicine through a vein to treat swelling in the brain caused by the immune system attacking itself.**
- strong intravenous medicine; Drug
- swelling in the brain; Disease
- immune system attacking itself; Disease

In [29]:
sentence = "The doctors started strong medicine through a vein to treat swelling in the brain caused by the immune system attacking itself."
gold_entities = [
    ['strong intravenous medicine', 'Drug'],
    ['swelling in the brain', 'Disease'],
    ['immune system attacking itself', 'Disease']
]

cde, exh_cde, ef, cdef_05, cdef_1, cdef_15 = run_test_on_use_case(types, sentence, gold_entities)

SENTENCE:            The doctors started strong medicine through a vein to treat swelling in the brain caused by the immune system attacking itself.
GOLD_ENTITIES:       [['strong intravenous medicine', 'Drug'], ['swelling in the brain', 'Disease'], ['immune system attacking itself', 'Disease']]
GENERATED_ENTITIES:  [['swelling', 'Symptom'], ['strong medicine', 'Drug'], ['the immune system attacking itself', 'Disease']]
CDE:      0.7233145943836132
Exh_CDE   0.6666666666666666
EF:       0.0
CDEF-0.5  0.6881150700892814
CDEF-1.0: 0.7792541837724735
CDEF-1.5: 0.8515529326908527
